# Anomaly Detection for FR 2052a Regulatory Reporting

## Overview
This notebook provides a scalable framework for detecting anomalies in financial data from ClickHouse tables.

**Focus Areas:**
- **Outliers**: Extreme values that deviate significantly from normal patterns
- **Miscategorizations**: Data points that don't fit expected patterns for their regulatory bucket

## Key Features
- **Flexible Table Selection**: Easily switch between different tables to analyze
- **Flexible Column Selection**: Choose which columns to analyze
- **Scalable Processing**: Handles millions of rows through batch processing
- **Multiple Detection Methods**: Statistical, ML-based, and deep learning approaches
- **Memory Efficient**: Optimized for large datasets
- **Well Documented**: Extensive comments throughout

## Methods Included
1. **Statistical Methods**: Z-score and IQR for outlier detection
2. **Isolation Forest**: Tree-based anomaly detection
3. **Autoencoder (TensorFlow)**: Deep learning reconstruction-based detection
4. **DBSCAN**: Density-based clustering for anomaly detection

## Prerequisites
```bash
pip install -r requirements.txt
```

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
from clickhouse_driver import Client
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Configure ClickHouse Connection

**IMPORTANT**: Replace the placeholder credentials with your actual database credentials.

In [ ]:
ch = Client(
    host='sd-mccg-kpyd.dam.nsroot.net',
    port=9000,
    database='dcw_glrs_clean',
    user='YOUR_USERNAME',
    password='YOUR_PASSWORD',
    client_name='anomaly_detection',
    alt_hosts='sd-t2i1-7o4d.nam.nsroot.net:9000,sd-ybfb-pfe0.nam.nsroot.net:9000',
    settings={'use_numpy': True}
)

print("Database connection configured.")

### 2.1 Configure Target Table

**Set the table name you want to analyze.** Change this to analyze different tables in the database.

In [ ]:
TABLE_NAME = 'tb_elp_dep_dl'

print(f"Target table: {TABLE_NAME}")

## 3. Data Exploration Functions

These helper functions let you explore the table schema and sample data before selecting columns for analysis.

In [ ]:
def get_table_schema(client, table_name):
    query = f"DESCRIBE TABLE {table_name}"
    schema_df = client.query_dataframe(query)
    return schema_df

def get_table_count(client, table_name):
    query = f"SELECT COUNT(*) as count FROM {table_name}"
    result = client.query_dataframe(query)
    return result['count'].iloc[0]

def get_sample_data(client, table_name, sample_size=1000):
    query = f"SELECT * FROM {table_name} LIMIT {sample_size}"
    return client.query_dataframe(query)

print("Data exploration functions defined.")

### 3.1 Explore Table Schema

In [ ]:
schema = get_table_schema(ch, TABLE_NAME)
print("Table Schema:")
print(schema)

row_count = get_table_count(ch, TABLE_NAME)
print(f"\nTotal rows in table: {row_count:,}")

### 3.2 View Sample Data

In [ ]:
sample_df = get_sample_data(ch, TABLE_NAME, sample_size=1000)
print(f"Sample data shape: {sample_df.shape}")
print("\nFirst few rows:")
sample_df.head()

In [ ]:
print("Basic statistics for sample data:")
sample_df.describe()

## 4. Column Selection & Configuration

**Configure which columns to analyze for anomalies.**

Based on the schema above, populate these lists:
- **NUMERIC_COLUMNS**: Financial metrics to analyze (amounts, balances, rates, etc.)
- **CATEGORICAL_COLUMNS**: Categories for grouping/filtering (bucket types, classifications, etc.)
- **ID_COLUMNS**: Identifiers to track anomalies back to source (record IDs, transaction IDs, etc.)

In [ ]:
NUMERIC_COLUMNS = [

]

CATEGORICAL_COLUMNS = [

]

ID_COLUMNS = [

]

BATCH_SIZE = 100000

print(f"Numeric columns for analysis: {NUMERIC_COLUMNS}")
print(f"Categorical columns: {CATEGORICAL_COLUMNS}")
print(f"Identifier columns: {ID_COLUMNS}")
print(f"Batch size for processing: {BATCH_SIZE:,} rows")

## 5. Scalable Data Loading

This batch loading function handles millions of rows without memory issues.

In [ ]:
def load_data_in_batches(client, table_name,
                        columns=None, batch_size=100000,
                        max_rows=None, where_clause=None):
    if columns:
        col_str = ', '.join(columns)
    else:
        col_str = '*'

    where_sql = f"WHERE {where_clause}" if where_clause else ""

    total_query = f"SELECT COUNT(*) as count FROM {table_name} {where_sql}"
    total_rows = client.query_dataframe(total_query)['count'].iloc[0]

    if max_rows:
        total_rows = min(total_rows, max_rows)

    print(f"Loading {total_rows:,} rows in batches of {batch_size:,}...")

    offset = 0
    batch_num = 1

    while offset < total_rows:
        current_batch_size = min(batch_size, total_rows - offset)

        query = f"""
        SELECT {col_str}
        FROM {table_name}
        {where_sql}
        LIMIT {current_batch_size}
        OFFSET {offset}
        """

        batch_df = client.query_dataframe(query)

        print(f"Batch {batch_num}: Loaded {len(batch_df):,} rows (offset: {offset:,})")

        yield batch_df

        offset += current_batch_size
        batch_num += 1

print("Batch loading function defined.")

## 6. Data Preprocessing

Clean and normalize data before anomaly detection.

In [ ]:
def preprocess_data(df, numeric_cols, handle_missing='drop', scaler_type='standard'):
    df_copy = df[numeric_cols].copy()

    df_copy.replace([np.inf, -np.inf], np.nan, inplace=True)

    initial_rows = len(df_copy)

    if handle_missing == 'drop':
        df_copy.dropna(inplace=True)
    elif handle_missing == 'mean':
        df_copy.fillna(df_copy.mean(), inplace=True)
    elif handle_missing == 'median':
        df_copy.fillna(df_copy.median(), inplace=True)
    elif handle_missing == 'zero':
        df_copy.fillna(0, inplace=True)

    rows_after = len(df_copy)
    if initial_rows != rows_after:
        print(f"Removed {initial_rows - rows_after:,} rows with missing values")

    scaler = None
    if scaler_type == 'standard':
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_copy)
    elif scaler_type == 'robust':
        scaler = RobustScaler()
        scaled_data = scaler.fit_transform(df_copy)
    elif scaler_type == 'minmax':
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(df_copy)
    else:
        scaled_data = df_copy.values

    processed_df = pd.DataFrame(scaled_data, columns=numeric_cols, index=df_copy.index)

    return processed_df, scaler

print("Preprocessing function defined.")

## 7. Anomaly Detection Methods

### Method 1: Statistical Outlier Detection (Z-Score & IQR)

**Fast, interpretable method for detecting extreme values.**

- **Z-Score**: Measures how many standard deviations a point is from the mean
- **IQR (Interquartile Range)**: Detects outliers based on quartile ranges

**Best for**: Quick initial screening, univariate outliers

In [ ]:
def detect_statistical_anomalies(df, numeric_cols, method='zscore', threshold=3):
    data = df[numeric_cols]

    if method == 'zscore':
        z_scores = np.abs((data - data.mean()) / data.std())
        is_anomaly = (z_scores > threshold).any(axis=1)

    elif method == 'iqr':
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR

        is_anomaly = ((data < lower_bound) | (data > upper_bound)).any(axis=1)

    return is_anomaly

print("Statistical anomaly detection function defined.")

### Method 2: Isolation Forest

**Tree-based ensemble method that isolates anomalies efficiently.**

Works by randomly partitioning data - anomalies require fewer splits to isolate.

**Best for**: Multi-dimensional outliers, when you know approximate anomaly rate

In [ ]:
def detect_isolation_forest_anomalies(df, numeric_cols, contamination=0.01,
                                     n_estimators=100, random_state=42):
    data = df[numeric_cols].values

    model = IsolationForest(
        contamination=contamination,
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1
    )

    anomaly_labels = model.fit_predict(data)

    anomaly_scores = model.score_samples(data)

    return anomaly_labels, anomaly_scores, model

print("Isolation Forest function defined.")

### Method 3: Autoencoder (TensorFlow Deep Learning)

**Neural network that learns to reconstruct normal data patterns.**

The autoencoder compresses data to a lower-dimensional representation, then reconstructs it.
Normal data reconstructs well; anomalies have high reconstruction error.

**Architecture:**
- Encoder: Compresses input to lower dimensional representation
- Decoder: Reconstructs input from compressed representation

**Best for**: Complex patterns, sufficient training data (10,000+ records)

In [ ]:
def build_autoencoder(input_dim, encoding_dim=None, hidden_layers=[64, 32]):
    if encoding_dim is None:
        encoding_dim = max(input_dim // 2, 8)

    input_layer = layers.Input(shape=(input_dim,))

    encoded = input_layer
    for units in hidden_layers:
        encoded = layers.Dense(units, activation='relu')(encoded)
        encoded = layers.Dropout(0.2)(encoded)

    encoded = layers.Dense(encoding_dim, activation='relu', name='encoding')(encoded)

    decoded = encoded
    for units in reversed(hidden_layers):
        decoded = layers.Dense(units, activation='relu')(decoded)
        decoded = layers.Dropout(0.2)(decoded)

    decoded = layers.Dense(input_dim, activation='linear')(decoded)

    autoencoder = keras.Model(inputs=input_layer, outputs=decoded)

    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )

    return autoencoder

def train_autoencoder(model, X_train, epochs=50, batch_size=256,
                     validation_split=0.1, verbose=1):
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )

    history = model.fit(
        X_train, X_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        callbacks=[early_stopping, reduce_lr],
        verbose=verbose
    )

    return history

def detect_autoencoder_anomalies(model, X_test, threshold_percentile=95):
    reconstructions = model.predict(X_test, batch_size=256, verbose=0)

    reconstruction_errors = np.mean(np.square(X_test - reconstructions), axis=1)

    threshold = np.percentile(reconstruction_errors, threshold_percentile)

    anomaly_labels = reconstruction_errors > threshold

    return anomaly_labels, reconstruction_errors, threshold

print("Autoencoder functions defined.")

### Method 4: DBSCAN (Density-Based Clustering)

**Identifies anomalies as points in low-density regions.**

DBSCAN groups together points that are closely packed and marks isolated points as anomalies.

**Best for**: When anomalies are isolated clusters or in low-density regions

In [ ]:
def detect_dbscan_anomalies(df, numeric_cols, eps=0.5, min_samples=5):
    data = df[numeric_cols].values

    dbscan = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)

    cluster_labels = dbscan.fit_predict(data)

    is_anomaly = cluster_labels == -1

    return cluster_labels, is_anomaly

print("DBSCAN function defined.")

## 8. Visualization Functions

Functions to visualize and analyze anomaly detection results.

In [ ]:
def plot_anomaly_distribution(anomaly_labels, scores=None, title="Anomaly Distribution"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if isinstance(anomaly_labels[0], (bool, np.bool_)):
        anomaly_count = np.sum(anomaly_labels)
        normal_count = len(anomaly_labels) - anomaly_count
    else:
        anomaly_count = np.sum(anomaly_labels == -1)
        normal_count = np.sum(anomaly_labels == 1)

    axes[0].bar(['Normal', 'Anomaly'], [normal_count, anomaly_count],
               color=['green', 'red'], alpha=0.7)
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'{title}\nTotal: {len(anomaly_labels):,}')
    axes[0].set_yscale('log')

    for i, (label, count) in enumerate([('Normal', normal_count), ('Anomaly', anomaly_count)]):
        axes[0].text(i, count, f'{count:,}\n({count/len(anomaly_labels)*100:.2f}%)',
                    ha='center', va='bottom')

    if scores is not None:
        axes[1].hist(scores, bins=50, alpha=0.7, edgecolor='black')
        axes[1].set_xlabel('Anomaly Score')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Distribution of Anomaly Scores')
        axes[1].axvline(np.percentile(scores, 95), color='red',
                       linestyle='--', label='95th percentile')
        axes[1].legend()
    else:
        axes[1].axis('off')

    plt.tight_layout()
    plt.show()

def plot_pca_anomalies(df, numeric_cols, anomaly_labels, title="PCA Visualization"):
    data = df[numeric_cols].values

    pca = PCA(n_components=2)
    data_2d = pca.fit_transform(data)

    if isinstance(anomaly_labels[0], (bool, np.bool_)):
        is_anomaly = anomaly_labels
    else:
        is_anomaly = anomaly_labels == -1

    plt.figure(figsize=(10, 8))

    plt.scatter(data_2d[~is_anomaly, 0], data_2d[~is_anomaly, 1],
               c='green', alpha=0.3, label='Normal', s=10)

    plt.scatter(data_2d[is_anomaly, 0], data_2d[is_anomaly, 1],
               c='red', alpha=0.7, label='Anomaly', s=50, edgecolors='black')

    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'], label='Training Loss')
    axes[0].plot(history.history['val_loss'], label='Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MSE Loss')
    axes[0].set_title('Training History - Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['mae'], label='Training MAE')
    axes[1].plot(history.history['val_mae'], label='Validation MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].set_title('Training History - MAE')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print("Visualization functions defined.")

## 9. Complete Analysis Pipeline

### 9.1 Load Data

In [ ]:
all_columns = ID_COLUMNS + CATEGORICAL_COLUMNS + NUMERIC_COLUMNS

print(f"Loading data with columns: {all_columns}")

data_batches = []
for batch in load_data_in_batches(ch, TABLE_NAME, columns=all_columns, batch_size=BATCH_SIZE, max_rows=500000):
    data_batches.append(batch)

full_data = pd.concat(data_batches, ignore_index=True)
print(f"\nTotal data loaded: {len(full_data):,} rows")
print(f"Memory usage: {full_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

### 9.2 Preprocess Data

In [ ]:
processed_data, scaler = preprocess_data(
    full_data,
    NUMERIC_COLUMNS,
    handle_missing='drop',
    scaler_type='robust'
)

print(f"Processed data shape: {processed_data.shape}")

### 9.3 Statistical Methods

In [ ]:
zscore_anomalies = detect_statistical_anomalies(
    processed_data,
    NUMERIC_COLUMNS,
    method='zscore',
    threshold=3
)

print(f"Z-score anomalies detected: {zscore_anomalies.sum():,} ({zscore_anomalies.sum()/len(zscore_anomalies)*100:.2f}%)")

plot_anomaly_distribution(zscore_anomalies, title="Z-Score Anomaly Detection")

In [ ]:
iqr_anomalies = detect_statistical_anomalies(
    processed_data,
    NUMERIC_COLUMNS,
    method='iqr',
    threshold=1.5
)

print(f"IQR anomalies detected: {iqr_anomalies.sum():,} ({iqr_anomalies.sum()/len(iqr_anomalies)*100:.2f}%)")

plot_anomaly_distribution(iqr_anomalies, title="IQR Anomaly Detection")

### 9.4 Isolation Forest

In [ ]:
if_labels, if_scores, if_model = detect_isolation_forest_anomalies(
    processed_data,
    NUMERIC_COLUMNS,
    contamination=0.01,
    n_estimators=100
)

print(f"Isolation Forest anomalies: {np.sum(if_labels == -1):,}")

plot_anomaly_distribution(if_labels, if_scores, title="Isolation Forest")

In [ ]:
plot_pca_anomalies(processed_data, NUMERIC_COLUMNS, if_labels,
                  title="Isolation Forest Anomalies (PCA Visualization)")

### 9.5 Autoencoder (Deep Learning)

In [ ]:
input_dim = len(NUMERIC_COLUMNS)
print(f"Building autoencoder with input dimension: {input_dim}")

autoencoder = build_autoencoder(
    input_dim=input_dim,
    encoding_dim=max(input_dim // 2, 8),
    hidden_layers=[64, 32]
)

autoencoder.summary()

In [ ]:
X_train = processed_data.values

history = train_autoencoder(
    autoencoder,
    X_train,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

plot_training_history(history)

In [ ]:
ae_anomalies, ae_errors, ae_threshold = detect_autoencoder_anomalies(
    autoencoder,
    X_train,
    threshold_percentile=95
)

print(f"Autoencoder anomalies: {ae_anomalies.sum():,} ({ae_anomalies.sum()/len(ae_anomalies)*100:.2f}%)")
print(f"Reconstruction error threshold: {ae_threshold:.6f}")

plot_anomaly_distribution(ae_anomalies, ae_errors, title="Autoencoder Anomaly Detection")

In [ ]:
plot_pca_anomalies(processed_data, NUMERIC_COLUMNS, ae_anomalies,
                  title="Autoencoder Anomalies (PCA Visualization)")

## 10. Combine Results & Export

Combine results from all methods and export flagged anomalies.

In [ ]:
results_df = full_data.loc[processed_data.index].copy()

results_df['zscore_anomaly'] = zscore_anomalies.values
results_df['iqr_anomaly'] = iqr_anomalies.values
results_df['isolation_forest_anomaly'] = if_labels == -1
results_df['isolation_forest_score'] = if_scores
results_df['autoencoder_anomaly'] = ae_anomalies
results_df['autoencoder_error'] = ae_errors

results_df['anomaly_count'] = (
    results_df['zscore_anomaly'].astype(int) +
    results_df['iqr_anomaly'].astype(int) +
    results_df['isolation_forest_anomaly'].astype(int) +
    results_df['autoencoder_anomaly'].astype(int)
)

results_df['is_anomaly'] = results_df['anomaly_count'] >= 2

print(f"\nAnomalies flagged by multiple methods:")
print(results_df['anomaly_count'].value_counts().sort_index())

anomalies_only = results_df[results_df['is_anomaly']].copy()
print(f"\nTotal anomalies (2+ methods): {len(anomalies_only):,}")

anomalies_only.to_csv('anomalies_detected.csv', index=False)
print("\nAnomalies saved to: anomalies_detected.csv")

anomalies_only.head(20)

## 11. Analyze Top Anomalies

In [ ]:
top_anomalies = anomalies_only.nlargest(20, 'anomaly_count')

print("Top 20 Most Severe Anomalies:")
print("="*80)
for idx, row in top_anomalies.iterrows():
    print(f"\nRecord: {row[ID_COLUMNS].to_dict() if ID_COLUMNS else idx}")
    print(f"Flagged by {row['anomaly_count']}/4 methods")
    print(f"Isolation Forest Score: {row['isolation_forest_score']:.4f}")
    print(f"Autoencoder Error: {row['autoencoder_error']:.4f}")
    if CATEGORICAL_COLUMNS:
        print(f"Categories: {row[CATEGORICAL_COLUMNS].to_dict()}")
    print(f"Values: {row[NUMERIC_COLUMNS].to_dict()}")
    print("-"*80)

## 12. Save Models for Future Use

In [ ]:
import joblib

joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved to: scaler.pkl")

joblib.dump(if_model, 'isolation_forest_model.pkl')
print("Isolation Forest model saved to: isolation_forest_model.pkl")

autoencoder.save('autoencoder_model.h5')
print("Autoencoder saved to: autoencoder_model.h5")

print("\nAll models saved successfully!")

## 13. Summary and Recommendations

### Method Comparison

| Method | Speed | Interpretability | Best Use Case |
|--------|-------|-----------------|---------------|
| **Z-Score/IQR** | Very Fast | High | Quick screening, univariate outliers |
| **Isolation Forest** | Fast | Medium | Multi-dimensional outliers |
| **Autoencoder** | Moderate | Low | Complex patterns, sufficient data |
| **DBSCAN** | Slow | Medium | Low-density region anomalies |

### Recommendations for FR 2052a Compliance

1. **Multi-Method Approach**: Use consensus from multiple methods
   - Flag records detected by 2+ methods as high priority
   - Records flagged by all methods warrant immediate investigation

2. **Threshold Tuning**:
   - Start with conservative thresholds (Z-score=3, contamination=0.01)
   - Adjust based on investigation capacity and false positive rate

3. **Regular Retraining**:
   - Retrain models monthly or quarterly as patterns evolve
   - Track model performance over time

4. **Domain Expert Review**:
   - Always involve subject matter experts for final validation
   - Document investigation outcomes to improve future detection

5. **Workflow Integration**:
   - Automate anomaly detection in your reporting pipeline
   - Create alerts for new anomalies
   - Track metrics: detection rate, false positives, investigation time

### Parameter Tuning Guide

**Statistical Methods**:
- Z-score threshold: 2 (aggressive) to 4 (conservative)
- IQR multiplier: 1.5 (standard) to 3 (conservative)

**Isolation Forest**:
- Contamination: Set to expected anomaly rate (typically 0.01-0.05)
- N_estimators: 100-200 (more = slower but more stable)

**Autoencoder**:
- Threshold percentile: 90-99 (higher = fewer anomalies)
- Hidden layers: Adjust based on data complexity
- Epochs: 20-100 (use early stopping)

**DBSCAN**:
- eps: Requires experimentation (try 0.3-1.0)
- min_samples: 5-10 for typical datasets

### Next Steps

1. ✅ Fill in NUMERIC_COLUMNS, CATEGORICAL_COLUMNS, ID_COLUMNS
2. ✅ Run data exploration cells to understand your data
3. ✅ Execute the complete pipeline on a sample (500K rows)
4. ✅ Review detected anomalies with compliance team
5. ✅ Tune thresholds based on feedback
6. ✅ Scale up to full dataset
7. ✅ Integrate into regular workflow
8. ✅ Monitor and maintain over time